# 13 - Candidate recall (Gate A)

**Objective.** Before any ranker: is the truth **retrievable** from the candidate universe at a given mass tolerance? A ranker cannot recover a truth that candidate generation never proposes.

**Gate A is defined only under a valid C2 protocol** (`casmi.validation.c2_protocol.assess_c2_protocol`): the universe contains non-TRAIN structures, the regime table was rebuilt for *this* universe in `external_universe` mode (every C2 truth exists in a non-TRAIN source, all its reference spectra hidden), and C2 has queries. Otherwise the notebook persists `FAIL_PROTOCOL_INVALID` with the reason and reports **no** C2 recall number as C2 evidence -- a TRAIN-only universe can never appear to pass.

Reported:

* always: C1 / C2 / C3 counts, universe size, source counts / provenance, `C2_PROTOCOL_VALID` and warnings;
* only if valid (C2, `external_only` reachability = the truth counts only through a non-TRAIN source): Recall@All at 2 / 5 / 10 ppm, Recall@100 and Recall@25 at 5 ppm, candidate-count median / p90 / p99 at 5 ppm (`recall.gate_a` in the config);
* diagnostics valid in every mode (clearly labelled): `universe`-mode recall (the truth is always in the pool there, so recall < 1 is mass error alone), mass accuracy by instrument / adduct, and an **out-of-fold** mass-calibration analysis (offsets learned on training folds only, evaluated on the held-out fold).

Candidate order for Recall@k = retrieval order (abs ppm, then candidate_id). Fold views exclude that fold's C3 truths. CPU only; the mass index is staged to the local SSD.

**Writes:** `reports/candidate_recall/` (incl. `gate_a_decision.json`, `mass_calibration/`) and append-only runs in `runs/records/`.

In [ ]:
# >>> CASMI BOOTSTRAP casmi-v2-bootstrap-2 (generated by scripts/stamp_notebooks.py; do not edit by hand)
# mount Drive -> env roots -> clone if missing, else fetch + checkout + pull --ff-only -> drop stale casmi modules -> src on sys.path
import importlib, os, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', '/content/Enveda'))
os.environ.setdefault('ENVEDA_SCRATCH_DIR', '/content/enveda_work')
BRANCH = os.environ.setdefault('ENVEDA_BRANCH', 'main')
SYNC_REPO = os.environ.get('ENVEDA_SYNC_REPO', '1' if IN_COLAB else '0') == '1'  # outside Colab the working tree is used as is
GITHUB_REPO = 'mybenkhadda/enveda-kaggle'


def _git(*args):
    r = subprocess.run(['git', '-C', str(REPO_ROOT), *args], capture_output=True, text=True)
    if r.returncode != 0:
        cmd = ' '.join(args)
        raise RuntimeError(f'git {cmd} failed in {REPO_ROOT}: {r.stderr.strip()} | Fix: discard local edits inside the clone '
                           f'(git -C {REPO_ROOT} stash) or delete {REPO_ROOT}, then re-run this cell; source lives on GitHub only.')
    return r.stdout.strip()


if SYNC_REPO:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')  # optional Colab secret (private repository)
        except Exception:
            token = None
    url = f'https://{token}@github.com/{GITHUB_REPO}.git' if token else f'https://github.com/{GITHUB_REPO}.git'
    if not (REPO_ROOT / '.git').exists():
        r = subprocess.run(['git', 'clone', '--branch', BRANCH, url, str(REPO_ROOT)], capture_output=True, text=True)
        if r.returncode != 0:
            err = r.stderr.replace(token, '***') if token else r.stderr
            raise RuntimeError(f'git clone of {GITHUB_REPO} failed: {err.strip()} | Fix: check the GITHUB_TOKEN Colab secret / network.')
    else:
        _git('fetch', 'origin', BRANCH)
        _git('checkout', BRANCH)
        _git('pull', '--ff-only', 'origin', BRANCH)
if not (REPO_ROOT / 'src' / 'casmi' / 'workspace' / 'versions.py').exists():
    raise RuntimeError(f'{REPO_ROOT} has no src/casmi/workspace/versions.py: the source is OLDER than this notebook. '
                       f'Fix: push the bootstrap/artifact-registry refactor to GitHub {BRANCH}, restart the runtime, re-run this cell.')
for _m in [m for m in list(sys.modules) if m == 'casmi' or m.startswith('casmi.')]:
    del sys.modules[_m]  # after a pull: never mix old and new source in one kernel
importlib.invalidate_caches()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.environ['ENVEDA_BOOTSTRAP_CELL'] = 'casmi-v2-bootstrap-2'
_head = _git('log', '-1', '--format=%h %s') if (REPO_ROOT / '.git').exists() else 'no git (uploaded copy)'
print(f'source {REPO_ROOT} @ {_head} | drive {DRIVE_ROOT} | colab {IN_COLAB} | synced {SYNC_REPO}')

In [ ]:
from casmi.workspace.bootstrap import bootstrap

NOTEBOOK = '13_colab_candidate_recall'
CTX = bootstrap(NOTEBOOK, notebook_api='casmi-v2-notebooks-4',
                uses_paths=('repo_root', 'scratch_dir', 'experiments_dir'),
                uses_artifacts=('universe_root', 'candidate_recall_reports_dir', 'gate_a_decision', 'c2_recall_per_query', 'mass_calibration_dir'),
                requires=('dev_queries', 'validation_regimes', 'validation_regimes_meta', 'universe_manifest', 'candidate_keys', 'bucket_offsets',
                          'candidate_mass_index', 'universe_buckets'),
                signatures={'casmi.candidates.gate_a:recall_sweeps': ['queries', 'regimes', 'index', 'candidate_keys', 'has_ref', 'ppm_grid'],
                            'casmi.validation.c2_protocol:assess_c2_protocol': ['universe', 'regimes_meta', 'n_c2_queries'],
                            'casmi.validation.c2_protocol:gate_a_decision': ['protocol', 'summary', 'gate_cfg'],
                            'casmi.validation.leakage_audit:audit_gate_a': ['sweep', 'c3_retrieved_per_fold', 'protocol', 'gate_record', 'truth_rows'],
                            'casmi.validation.mass_calibration:cross_fold_offsets': ['df', 'fold_col', 'group_cols', 'min_group_size', 'n_bootstrap'],
                            'casmi.workspace.experiments:log_experiment': ['experiments_dir', 'experiment_id', 'description', 'channels', 'notebook',
                                                                          'status', 'decision', 'regime_identity', 'universe_identity']})
CFG, P, ARTIFACTS, MON = CTX.cfg, CTX.paths, CTX.artifacts, CTX.monitor
OUT = ARTIFACTS.candidate_recall_reports_dir
OUT.mkdir(parents=True, exist_ok=True)

## 1. Regimes and queries

In [ ]:
import json
import numpy as np, pandas as pd
from casmi.workspace.artifact_registry import read_json

REG = pd.read_parquet(ARTIFACTS.validation_regimes)
REG_META = read_json(ARTIFACTS.validation_regimes_meta)
dev = pd.read_parquet(ARTIFACTS.dev_queries, columns=['query_id', 'neutral_mass', 'exact_mass_true', 'ionization_mode', 'adduct', 'instrument_type'])
dev['query_id'] = dev['query_id'].astype(str)
Q = REG.dropna(subset=['regime']).merge(dev, on='query_id', how='left', validate='one_to_one')
N_BY_REGIME = {r: int((Q['regime'] == r).sum()) for r in ('C1', 'C2', 'C3')}
display(Q.groupby(['fold', 'regime']).size().unstack(fill_value=0))
print('queries per regime:', N_BY_REGIME, '| regime c2_mode:', REG_META.get('c2_mode'))

## 2. C2 protocol validity (decided before any recall number)

In [ ]:
from casmi.validation.c2_protocol import assess_c2_protocol, assess_universe

UNIVERSE = assess_universe(ARTIFACTS, CFG)
PROTOCOL = assess_c2_protocol(UNIVERSE, REG_META, N_BY_REGIME['C2'])
C2_PROTOCOL_VALID = PROTOCOL['protocol_valid']
S = UNIVERSE.source_summary or {}
OVERVIEW = pd.Series({'C1 queries': N_BY_REGIME['C1'], 'C2 queries': N_BY_REGIME['C2'], 'C3 queries': N_BY_REGIME['C3'],
                      'universe size': UNIVERSE.n_candidates, 'TRAIN-only': S.get('train_only'), 'external-only': S.get('external_only'),
                      'TRAIN+external': S.get('train_and_external'), 'universe status': UNIVERSE.universe_status,
                      'regime c2_mode': PROTOCOL['c2_mode'], 'C2_PROTOCOL_VALID': C2_PROTOCOL_VALID}, name='value').to_frame()
display(OVERVIEW)
display(pd.Series(S.get('per_source', {}), name='n_connectivities').to_frame())
for r in PROTOCOL['reasons']:
    print('WARNING:', r)
print('C2_PROTOCOL_VALID =', C2_PROTOCOL_VALID)

## 3. Stage the index to the local SSD and attach truth provenance

In [ ]:
from casmi.candidates.gate_a import attach_truth_provenance, reference_flags
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.workspace.staging import stage_directory, stage_file

with MON.stage('stage index'):
    IDX_DIR, staged = stage_directory(ARTIFACTS.candidate_mass_index, P.scratch_dir, rel='universe/index')
    IDX = CandidateMassIndex.load(ARTIFACTS.candidate_mass_index if staged['not_staged'] else IDX_DIR)
    UKEYS = np.load(stage_file(ARTIFACTS.candidate_keys, P.scratch_dir, rel='universe/candidate_keys.npy'), mmap_mode='r')
Q = attach_truth_provenance(Q, ARTIFACTS.universe_root, UKEYS)
HAS_REF = reference_flags(ARTIFACTS.universe_root, IDX.n_candidates)
display(Q.groupby('regime')[['reach_universe', 'reach_external']].mean())
print(f'universe candidates: {IDX.n_candidates:,} | truths in universe: {(Q.truth_id >= 0).mean():.4f} | candidates with reference spectra: {HAS_REF.mean():.4f}')

## 4. Recall sweeps per fold view

For each fold: that fold's C3 truths are removed from the pool and its hidden (C2/C3) truths lose their reference flag; C1 + C2 queries are swept in `external_only` and `universe` modes; C3 truths are checked to be never retrievable.

In [ ]:
from casmi.candidates.gate_a import per_fold_recall, recall_sweeps
from casmi.candidates.recall import recall_by_ppm

GA = CFG['recall']['gate_a']
PPM = sorted(set(CFG['recall']['ppm_grid']) | set(GA['recall_all_ppm']) | {GA['recall_at_k_ppm'], GA['pool_quantiles_ppm'], GA['decision_ppm']})
K = sorted(set(CFG['recall']['k_values']) | {25, 100})
with MON.stage('recall sweeps', n_items=int(Q['regime'].isin(['C1', 'C2']).sum()) * len(PPM) * 2, unit='query-ppm'):
    SWEEP, C3_RETRIEVED = recall_sweeps(Q, REG, IDX, UKEYS, HAS_REF, PPM)
SUMMARY = recall_by_ppm(SWEEP, k_values=K, group_cols=('mode', 'regime'))
PER_FOLD = per_fold_recall(SWEEP, GA['decision_ppm'], K)
print('sweep rows:', len(SWEEP), '| C3 truths retrieved per fold (must be 0):', C3_RETRIEVED)

## 5. Gate A

`gate_a_decision` returns the Gate-A metrics only under a valid protocol; otherwise `FAIL_PROTOCOL_INVALID` with the reasons and `metrics = None`.

In [ ]:
from casmi.validation.c2_protocol import gate_a_decision

GATE_A = gate_a_decision(PROTOCOL, SUMMARY, GA)
print(json.dumps(GATE_A, indent=2, default=str))
if C2_PROTOCOL_VALID:
    display(pd.Series(GATE_A['metrics'], name='C2 external_only').to_frame())
    display(PER_FOLD[(PER_FOLD['mode'] == 'external_only') & (PER_FOLD['regime'] == 'C2')][['fold', 'n_queries', 'recall_all', 'recall_at_25', 'recall_at_100']])
else:
    print('Gate A =', GATE_A['decision'], '-- no C2 candidate-recall number is reported as C2 evidence.')

## 6. Diagnostics valid in every mode: universe-mode recall and mass accuracy

In `universe` mode the truth is always in the pool (TRAIN included), so recall below 1 is caused by mass error alone; these numbers are **not** C2 evidence. The hidden test is Bruker timsTOF: the timsTOF rows are the relevant operating point, other libraries can be much noisier. Errors > 1000 ppm indicate adduct / charge labelling errors.

In [ ]:
from casmi.candidates.gate_a import mass_accuracy

DIAG = SUMMARY[SUMMARY['mode'] == 'universe']
display(DIAG[['regime', 'ppm', 'n_queries', 'recall_all', 'recall_at_25', 'recall_at_100', 'pool_size_median', 'pool_size_p90', 'pool_size_p99']])
MASS_ACC = mass_accuracy(SWEEP, max(PPM))
for c, t in MASS_ACC.items():
    print('---', c)
    display(t)
TIMS_IDS = set(Q.loc[Q['instrument_type'].astype(str).str.lower().str.contains('timstof', na=False), 'query_id'])
U = SWEEP[SWEEP['mode'] == 'universe']
TIMS_RECALL = recall_by_ppm(U[U['query_id'].isin(TIMS_IDS)], k_values=K, group_cols=('regime',)) if TIMS_IDS else pd.DataFrame()
print(f'timsTOF queries: {len(TIMS_IDS):,} (universe mode, diagnostic)')
if len(TIMS_RECALL):
    display(TIMS_RECALL[['regime', 'ppm', 'n_queries', 'recall_all', 'recall_at_25', 'pool_size_median']])

## 7. Out-of-fold mass calibration (diagnostic)

Signed error `e = (observed neutral mass - true exact mass) / true * 1e6`. For each held-out fold, per-(instrument, adduct) median offsets are fitted on the **other folds only** (fallback to instrument, then global; gross errors > 50 ppm excluded from the fit), applied to the held-out fold, and the before / after recall and pool size are compared at the configured tolerances. Nothing is assumed: an offset is kept only if the held-out comparison shows smaller pools at equal or better recall.

In [ ]:
from casmi.validation.mass_calibration import apply_offset, calibration_effect, cross_fold_offsets, signed_ppm

MC = CFG['mass_calibration']
CAL = Q[Q['regime'].isin(['C1', 'C2']) & (Q['truth_id'] >= 0)].copy()
CAL['signed_ppm'] = signed_ppm(CAL['neutral_mass'], CAL['exact_mass_true'])
with MON.stage('mass calibration', n_items=len(CAL), unit='queries'):
    OFFSETS, OFFSET_TABLE = cross_fold_offsets(CAL, 'fold', MC['group_cols'], min_group_size=MC['min_group_size'], n_bootstrap=MC['n_bootstrap'])
    CAL['corrected_mass'] = apply_offset(CAL['neutral_mass'], OFFSETS)
    CAL_ALL = calibration_effect(IDX, CAL['neutral_mass'], CAL['corrected_mass'], CAL['truth_id'], MC['ppm_grid'], query_ids=CAL['query_id'].to_numpy())
    T = CAL[CAL['query_id'].isin(TIMS_IDS)]
    CAL_TIMS = (calibration_effect(IDX, T['neutral_mass'], T['corrected_mass'], T['truth_id'], MC['ppm_grid'], query_ids=T['query_id'].to_numpy())
                if len(T) else pd.DataFrame())
display(OFFSET_TABLE[OFFSET_TABLE['usable']].sort_values(['held_out_fold', 'n'], ascending=[True, False]).groupby('held_out_fold').head(8))
COLS = ['variant', 'ppm', 'n_queries', 'recall_all', 'recall_at_25', 'pool_size_median', 'pool_size_p90']
print('all C1/C2 queries (held-out folds):'); display(CAL_ALL[COLS])
if len(CAL_TIMS):
    print('timsTOF queries (held-out folds):'); display(CAL_TIMS[COLS])

## 8. Breakdowns of C2 recall

With a valid protocol: C2 `external_only` recall by mass, ion mode, adduct, formula class, truth source database and pool size (which chemistry the universe misses). Without one, the `universe` view is shown and labelled as a diagnostic.

In [ ]:
from casmi.validation.metrics import candidate_stage_metrics
from casmi.validation.reporting import add_strata, stratified_table

BREAKDOWN_MODE = 'external_only' if C2_PROTOCOL_VALID else 'universe'
c2 = SWEEP[(SWEEP['mode'] == BREAKDOWN_MODE) & (SWEEP['regime'] == 'C2')]
print('breakdown mode:', BREAKDOWN_MODE, '' if C2_PROTOCOL_VALID else '(DIAGNOSTIC -- not C2 evidence)')
BREAKDOWN = pd.DataFrame()
if len(c2):
    c2 = add_strata(c2.drop(columns=['neutral_mass', 'ionization_mode', 'adduct', 'instrument_type']), Q, CFG['validation']['mass_bins_da'],
                    CFG['validation']['pool_size_bins'], group_cols=('fold', 'true_connectivity_key', 'ppm'))
    BREAKDOWN = pd.concat([stratified_table(g, candidate_stage_metrics, strata=('mass_bin', 'ionization_mode', 'adduct', 'formula_class',
                                                                                'truth_sources', 'pool_size_bin'), k_values=K, min_queries=20).assign(ppm=ppm)
                           for ppm, g in c2.groupby('ppm')], ignore_index=True)
    display(BREAKDOWN[(BREAKDOWN['ppm'] == GA['recall_at_k_ppm']) & (BREAKDOWN['regime'] == 'C2')].sort_values(['stratum', 'recall_all']).head(60))

## 9. LEAKAGE AUDIT

`casmi.validation.leakage_audit.audit_gate_a` raises on failure: C3 truths are never retrievable in their own fold view (truth-structure removal); `external_only` reachability uses non-TRAIN provenance only (no candidate-source leakage into the metric); a protocol-invalid Gate A carries no C2 metrics. The C2 pool share with reference spectra is printed as a shortcut diagnostic (never a feature). Fold isolation and complete truth-reference hiding were asserted in notebook 11; queries are training spectra (the visible test set is never used).

In [ ]:
from casmi.validation.leakage_audit import audit_gate_a

AUDIT = audit_gate_a(SWEEP, C3_RETRIEVED, PROTOCOL, GATE_A, truth_rows=Q[['reach_external', 'truth_sources']])
display(AUDIT)

## 10. Persist and log (append-only)

In [ ]:
from casmi.validation.reporting import save_report
from casmi.workspace.artifacts import write_metadata
from casmi.workspace.bootstrap import write_json
from casmi.workspace.experiments import log_experiment

save_report(SUMMARY, {'notebook': NOTEBOOK, 'gate_a': GATE_A, 'protocol': PROTOCOL, 'universe': UNIVERSE.as_dict(),
                      'overview': OVERVIEW['value'].astype(str).to_dict(), 'ppm_grid': PPM, 'k_values': K, 'breakdown_mode': BREAKDOWN_MODE,
                      'leakage_audit': AUDIT.to_dict(orient='records'), 'resources': MON.table().to_dict(orient='records')},
            OUT, 'c2_candidate_recall')
write_json(ARTIFACTS.gate_a_decision, {**GATE_A, 'overview': OVERVIEW['value'].astype(str).to_dict(), 'universe_identity': UNIVERSE.identity,
                                       'regime_signature': REG_META.get('signature'), **CTX.metadata()})
SWEEP.to_parquet(ARTIFACTS.c2_recall_per_query, index=False)
PER_FOLD.to_parquet(OUT / 'candidate_recall_per_fold.parquet', index=False)
if len(BREAKDOWN):
    BREAKDOWN.to_parquet(OUT / 'c2_candidate_recall_breakdown.parquet', index=False)
pd.concat([t.reset_index().rename(columns={c: 'value'}).assign(stratum=c) for c, t in MASS_ACC.items()]).to_parquet(OUT / 'mass_accuracy.parquet', index=False)
ARTIFACTS.mass_calibration_dir.mkdir(parents=True, exist_ok=True)
OFFSET_TABLE.to_parquet(ARTIFACTS.mass_calibration_dir / 'oof_offsets.parquet', index=False)
CAL_ALL.to_parquet(ARTIFACTS.mass_calibration_dir / 'effect_all.parquet', index=False)
if len(CAL_TIMS):
    CAL_TIMS.to_parquet(ARTIFACTS.mass_calibration_dir / 'effect_timstof.parquet', index=False)
write_metadata(OUT, NOTEBOOK, config={'recall': CFG['recall'], 'mass_calibration': MC, 'c2_protocol': CFG.get('c2_protocol')}, seed=CTX.seed,
               repo_root=P.repo_root, input_paths={'regimes': ARTIFACTS.validation_regimes, 'universe': ARTIFACTS.universe_root}, extra=CTX.metadata())

In [ ]:
STATUS = 'COMPLETED' if C2_PROTOCOL_VALID else 'PROTOCOL_INVALID'
METRICS = GATE_A['metrics'] or {}
REC = METRICS.get('recall_all@' + str(GA['decision_ppm']) + 'ppm')
common = dict(git_commit=CTX.git_commit, notebook=NOTEBOOK, config=CFG['recall'], status=STATUS, decision='DIAGNOSTIC',
              regime_identity=REG_META.get('signature'), universe_identity=UNIVERSE.identity,
              candidate_generator='mass_index_' + str(GA['decision_ppm']) + 'ppm')
log_experiment(P.experiments_dir, 'gateA_c2_recall', 'Gate A: C2 external_only candidate recall (mass order)', ['mass'],
               config_path=CFG['_config_path'], candidate_recall=REC, n_queries=N_BY_REGIME['C2'], runtime=MON.elapsed(),
               notes=GATE_A['decision'] + ' | ' + str(GATE_A.get('reason', ''))[:300],
               artifact_paths={'gate_a_decision': ARTIFACTS.gate_a_decision, 'per_query': ARTIFACTS.c2_recall_per_query},
               extra={'gate_a_metrics': GATE_A['metrics']}, **common)
if C2_PROTOCOL_VALID:
    for f, g in PER_FOLD[(PER_FOLD['mode'] == 'external_only') & (PER_FOLD['regime'] == 'C2')].groupby('fold'):
        log_experiment(P.experiments_dir, 'gateA_c2_recall', 'Gate A per fold', ['mass'], fold=int(f), candidate_recall=float(g['recall_all'].iloc[0]),
                       n_queries=int(g['n_queries'].iloc[0]), **common)
print('saved', OUT, '| Gate A:', GATE_A['decision'], '| status', STATUS)
if C2_PROTOCOL_VALID:
    print('NEXT: 14_colab_analog_baseline')
else:
    print('NEXT: provide an external source (COCONUT) -> 12 -> 11 (REBUILD = True) -> 13. Do NOT start notebook 14 or any GPU work yet.')